# Train Pong
Training PPO model on Pong environment.

In [ ]:
!pip install torch gymnasium[atari] stable-baselines3 auto-rom.accept-rom-license numpy opencv-python

import gymnasium as gym
import ale_py
from gymnasium.wrappers import RecordVideo
import cv2
import numpy as np
from gymnasium import ObservationWrapper
from gymnasium.spaces import Box
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from gymnasium import spaces
import matplotlib.pyplot as plt
from collections import defaultdict
import seaborn as sns
import os

# PPO Framework Code
class PatchEmbedCNN(nn.Module):
    def __init__(self, input_channels=1, output_dim=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_channels, 32, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),
            nn.ReLU()
        )
        self.fc = nn.Linear(7 * 7 * 64, output_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)

class GeneralizedExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        input_channels = observation_space.shape[0]
        self.encoder = PatchEmbedCNN(input_channels=input_channels, output_dim=features_dim)

    def forward(self, observations):
        return self.encoder(observations)

def make_ppo_model(env, action_type='discrete'):
    policy_kwargs = dict(
        features_extractor_class=GeneralizedExtractor,
        net_arch=[128, 128]
    )
    model = PPO("CnnPolicy", env, policy_kwargs=policy_kwargs, verbose=1)
    return model

class TrainingTracker:
    def __init__(self, game_name):
        self.game_name = game_name
        self.history = defaultdict(list)
        self.episode_rewards = []
        self.episode_lengths = []
        self.wins = []
        self.advantages = []
        
    def update(self, reward, episode_length, advantage, win_status=None):
        self.episode_rewards.append(reward)
        self.episode_lengths.append(episode_length)
        self.advantages.append(advantage)
        if win_status is not None:
            self.wins.append(1 if win_status else 0)
            
    def log_step(self, step, policy_loss, value_loss, entropy):
        self.history['policy_loss'].append(policy_loss)
        self.history['value_loss'].append(value_loss)
        self.history['entropy'].append(entropy)
        self.history['steps'].append(step)
        
    def print_progress(self, episode, total_episodes):
        avg_reward = np.mean(self.episode_rewards[-100:]) if self.episode_rewards else 0
        avg_length = np.mean(self.episode_lengths[-100:]) if self.episode_lengths else 0
        avg_advantage = np.mean(self.advantages[-100:]) if self.advantages else 0
        win_rate = np.mean(self.wins[-100:]) * 100 if self.wins else 0
        
        print(f"\nGame: {self.game_name}")
        print(f"Episode {episode}/{total_episodes}")
        print(f"Average Reward (last 100): {avg_reward:.2f}")
        print(f"Average Episode Length (last 100): {avg_length:.2f}")
        print(f"Average Advantage (last 100): {avg_advantage:.2f}")
        if self.wins:
            print(f"Win Rate (last 100): {win_rate:.2f}%")
        print("-" * 50)

    def plot_training_results(self, save_path=None):
        plt.style.use('seaborn')
        fig = plt.figure(figsize=(15, 10))
        
        plt.subplot(2, 2, 1)
        rewards_smoothed = np.convolve(self.episode_rewards, 
                                     np.ones(100)/100, 
                                     mode='valid')
        plt.plot(rewards_smoothed, label='Smoothed Reward')
        plt.plot(self.episode_rewards, alpha=0.3, label='Raw Reward')
        plt.title(f'{self.game_name} - Reward Curve')
        plt.xlabel('Episode')
        plt.ylabel('Reward')
        plt.legend()
        
        if self.wins:
            plt.subplot(2, 2, 2)
            win_rate = np.convolve(self.wins, 
                                 np.ones(100)/100, 
                                 mode='valid') * 100
            plt.plot(win_rate)
            plt.title(f'{self.game_name} - Win Rate')
            plt.xlabel('Episode')
            plt.ylabel('Win Rate (%)')
        
        plt.subplot(2, 2, 3)
        advantages_smoothed = np.convolve(self.advantages, 
                                        np.ones(100)/100, 
                                        mode='valid')
        plt.plot(advantages_smoothed, label='Smoothed Advantage')
        plt.plot(self.advantages, alpha=0.3, label='Raw Advantage')
        plt.title(f'{self.game_name} - Advantage')
        plt.xlabel('Episode')
        plt.ylabel('Advantage')
        plt.legend()
        
        plt.subplot(2, 2, 4)
        plt.plot(self.history['policy_loss'], label='Policy Loss')
        plt.plot(self.history['value_loss'], label='Value Loss')
        plt.plot(self.history['entropy'], label='Entropy')
        plt.title(f'{self.game_name} - Training Losses')
        plt.xlabel('Training Step')
        plt.ylabel('Loss Value')
        plt.legend()
        
        plt.tight_layout()
        
        if save_path:
            os.makedirs(save_path, exist_ok=True)
            plt.savefig(f"{save_path}/{self.game_name}_training_plots.png")
        plt.show()

# Pong Environment
class ResizeAndGrayScale(ObservationWrapper):
    def __init__(self, env, shape=(84, 84)):
        super().__init__(env)
        self.shape = shape
        self.observation_space = Box(
            low=0, high=255, shape=(self.shape[0], self.shape[1], 1), dtype=np.uint8
        )

    def observation(self, obs):
        obs = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        obs = cv2.resize(obs, self.shape, interpolation=cv2.INTER_AREA)
        return np.expand_dims(obs, -1).astype(np.uint8)

def make_pong_env(record=False):
    env = gym.make("ALE/Pong-v5", render_mode="rgb_array")
    env = ResizeAndGrayScale(env)
    if record:
        env = RecordVideo(env, video_folder="/content/videos", episode_trigger=lambda ep: True)
    return env

# Training Function
def train_ppo_with_tracking(env, model, total_timesteps, game_name="Pong"):
    tracker = TrainingTracker(game_name)
    timestep = 0
    episode = 0
    
    while timestep < total_timesteps:
        episode += 1
        obs = env.reset()[0]
        episode_reward = 0
        episode_steps = 0
        done = False
        
        while not done:
            action, _ = model.predict(obs)
            next_obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            episode_reward += reward
            episode_steps += 1
            timestep += 1
            
            obs = next_obs
            
            model.train()
            
        advantage = episode_reward
        win_status = episode_reward > 0
        
        tracker.update(episode_reward, episode_steps, advantage, win_status)
        tracker.log_step(timestep, 
                        model.policy.optimizer_loss if hasattr(model.policy, 'optimizer_loss') else 0,
                        model.value_loss if hasattr(model, 'value_loss') else 0,
                        model.entropy_loss if hasattr(model, 'entropy_loss') else 0)
        
        if episode % 10 == 0:
            tracker.print_progress(episode, total_timesteps//episode_steps)
            
    tracker.plot_training_results(save_path="/content/training_plots")
    return model

# Run Training
os.makedirs('/content/training_plots', exist_ok=True)
env = make_pong_env(record=False)
model = make_ppo_model(env, action_type='discrete')
model = train_ppo_with_tracking(env, model, total_timesteps=100000)  # Reduced for Colab
model.save("ppo_pong")